# Task 1: FedSGD vs Centralized SGD
**Goal: Demonstrate theoretical equivalence between FedSGD and centralized training**


## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset

from scriptsfl.data_utils import load_dataset, create_iid_split, get_dataloaders
from scriptsfl.models import SimpleCNN, get_model
from scriptsfl.federated_utils import get_model_weights, set_model_weights, aggregate_weights, evaluate_model
from scriptsfl.results_utils import save_results

torch.manual_seed(42)
np.random.seed(42)

print("="*80)
print("TASK 1: FEDSGD VS CENTRALIZED SGD")
print("="*80)

#%% Configuration
CONFIG = {
    'dataset': 'cifar10',
    'num_clients': 5,
    'num_iterations': 20,  # Number of SGD steps (NOT rounds/epochs)
    'batch_size': 64,
    'lr': 0.01,
    'momentum': 0.9,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}



In [ ]:
#%% Load data
train_dataset, test_dataset = load_dataset(CONFIG['dataset'])
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Create IID split for clients
client_indices = create_iid_split(train_dataset, CONFIG['num_clients'])
client_loaders = get_dataloaders(train_dataset, client_indices,
                                  batch_size=CONFIG['batch_size'], shuffle=False)
# NOTE: shuffle=False ensures deterministic ordering for comparison

## PART 1: FedSGD Implementation
FedSGD: Each client does 1 gradient step, then aggregates
This is equivalent to computing gradient on all data and taking 1 step

In [ ]:
print("\n--- Running FedSGD (K=1) ---")

# Initialize model for FedSGD
model_fedsgd = get_model('simplecnn', num_classes=10, dataset=CONFIG['dataset'])
model_fedsgd = model_fedsgd.to(CONFIG['device'])

# History tracking
fedsgd_history = {
    'iterations': [],
    'test_accuracy': [],
    'test_loss': [],
    'model_norm': []  # Track ||θ|| to verify equivalence
}

# FedSGD Training Loop
# TODO: For each iteration:
#   1. Distribute current global model to all clients
#   2. Each client computes gradient on ONE BATCH (or full local data)
#   3. Each client takes ONE SGD step
#   4. Server aggregates client models (weighted by data size)
#   5. Evaluate and record metrics

# Example structure:
for iteration in range(CONFIG['num_iterations']):
    # TODO: Implement FedSGD logic here
    # Hint: Create optimizers for each client, do 1 step, aggregate

    # For each client:
    #   - Get one batch from client's data
    #   - Compute loss and gradients
    #   - Take one optimizer step
    #   - Collect updated weights

    # Aggregate all client weights
    # Update global model

    # Evaluate
    test_acc, test_loss = evaluate_model(model_fedsgd, test_loader, CONFIG['device'])

    # Track model norm (for comparison)
    model_norm = sum(torch.norm(p).item() for p in model_fedsgd.parameters())

    fedsgd_history['iterations'].append(iteration + 1)
    fedsgd_history['test_accuracy'].append(test_acc)
    fedsgd_history['test_loss'].append(test_loss)
    fedsgd_history['model_norm'].append(model_norm)

    print(f"Iteration {iteration+1}: Test Acc = {test_acc:.2f}%, Loss = {test_loss:.4f}")

## PART 2: Centralized SGD Implementation
Centralized: Combine all data, compute gradient, take 1 step

In [ ]:
print("\n--- Running Centralized SGD ---")

# Combine all client data into one dataset
all_data = ConcatDataset([client_loaders[i].dataset for i in range(CONFIG['num_clients'])])
centralized_loader = DataLoader(all_data, batch_size=CONFIG['batch_size'], shuffle=False)
# NOTE: Use same batch size and no shuffling for fair comparison

# Initialize model for Centralized (same initialization as FedSGD)
torch.manual_seed(42)  # Reset seed to get same initialization
model_centralized = get_model('simplecnn', num_classes=10, dataset=CONFIG['dataset'])
model_centralized = model_centralized.to(CONFIG['device'])

# Copy exact same initial weights from FedSGD
# TODO: Copy initial weights from model_fedsgd to ensure identical starting point

# History tracking
centralized_history = {
    'iterations': [],
    'test_accuracy': [],
    'test_loss': [],
    'model_norm': []
}

# Centralized Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_centralized.parameters(), lr=CONFIG['lr'], momentum=CONFIG['momentum'])

# TODO: For each iteration:
#   1. Get one batch from centralized dataset
#   2. Compute loss and gradients
#   3. Take ONE optimizer step
#   4. Evaluate and record metrics

data_iter = iter(centralized_loader)
for iteration in range(CONFIG['num_iterations']):
    # TODO: Implement centralized SGD logic here

    # Get batch
    try:
        data, target = next(data_iter)
    except StopIteration:
        data_iter = iter(centralized_loader)
        data, target = next(data_iter)

    data, target = data.to(CONFIG['device']), target.to(CONFIG['device'])

    # One SGD step
    optimizer.zero_grad()
    output = model_centralized(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    # Evaluate
    test_acc, test_loss = evaluate_model(model_centralized, test_loader, CONFIG['device'])

    # Track model norm
    model_norm = sum(torch.norm(p).item() for p in model_centralized.parameters())

    centralized_history['iterations'].append(iteration + 1)
    centralized_history['test_accuracy'].append(test_acc)
    centralized_history['test_loss'].append(test_loss)
    centralized_history['model_norm'].append(model_norm)

    print(f"Iteration {iteration+1}: Test Acc = {test_acc:.2f}%, Loss = {test_loss:.4f}")

## PART 3: Comparison and Verification

In [ ]:
print("\n" + "="*80)
print("COMPARING FEDSGD VS CENTRALIZED")
print("="*80)

# Compute difference in model parameters
param_diff = 0.0
for p1, p2 in zip(model_fedsgd.parameters(), model_centralized.parameters()):
    param_diff += torch.norm(p1 - p2).item()

print(f"\nTotal parameter difference: {param_diff:.6f}")
print("(Should be very close to 0 if implementations are equivalent)")

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Test Accuracy
axes[0, 0].plot(fedsgd_history['iterations'], fedsgd_history['test_accuracy'],
                marker='o', label='FedSGD', linewidth=2)
axes[0, 0].plot(centralized_history['iterations'], centralized_history['test_accuracy'],
                marker='s', label='Centralized', linewidth=2, linestyle='--')
axes[0, 0].set_xlabel('Iterations')
axes[0, 0].set_ylabel('Test Accuracy (%)')
axes[0, 0].set_title('Test Accuracy Comparison')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Test Loss
axes[0, 1].plot(fedsgd_history['iterations'], fedsgd_history['test_loss'],
                marker='o', label='FedSGD', linewidth=2)
axes[0, 1].plot(centralized_history['iterations'], centralized_history['test_loss'],
                marker='s', label='Centralized', linewidth=2, linestyle='--')
axes[0, 1].set_xlabel('Iterations')
axes[0, 1].set_ylabel('Test Loss')
axes[0, 1].set_title('Test Loss Comparison')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Model Norm
axes[1, 0].plot(fedsgd_history['iterations'], fedsgd_history['model_norm'],
                marker='o', label='FedSGD', linewidth=2)
axes[1, 0].plot(centralized_history['iterations'], centralized_history['model_norm'],
                marker='s', label='Centralized', linewidth=2, linestyle='--')
axes[1, 0].set_xlabel('Iterations')
axes[1, 0].set_ylabel('Model Norm')
axes[1, 0].set_title('Model Parameter Norm Comparison')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Difference plot
acc_diff = [abs(a1 - a2) for a1, a2 in zip(fedsgd_history['test_accuracy'],
                                            centralized_history['test_accuracy'])]
axes[1, 1].plot(fedsgd_history['iterations'], acc_diff, marker='o', linewidth=2, color='red')
axes[1, 1].set_xlabel('Iterations')
axes[1, 1].set_ylabel('Absolute Accuracy Difference (%)')
axes[1, 1].set_title('Accuracy Difference (FedSGD - Centralized)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./results/task1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save results
results_task1 = {
    'fedsgd': fedsgd_history,
    'centralized': centralized_history,
    'param_difference': param_diff,
    'config': CONFIG
}
save_results(results_task1, 'task1_fedsgd_vs_centralized')

print("\n" + "="*80)
print("TASK 1 COMPLETE")
print("="*80)
print("\nConclusion:")
print("FedSGD with K=1 and full client participation is theoretically equivalent")
print("to centralized SGD when using the same learning rate and data ordering.")
print("Any differences observed should be minimal (due to floating point operations).")